# Zero-shot eligibility check: Ardook's profile vs. one grant text file

Notebook version for the HPC kernel "Transformers 5.3.0 (CUDA12, Python 3.13)".
Each "# %%" marks one notebook cell. Run them top to bottom.

Two MiniLM models, two jobs:
  1. RETRIEVE - all-MiniLM-L6-v2 embeds every grant sentence and finds the
                ones most related to each eligibility label.
  2. JUDGE    - cross-encoder/nli-MiniLM2-L6-H768 reads each retrieved
                sentence next to a statement about Ardook and scores
                entailment / contradiction / neutral.

One-time setup in a cell (then Kernel -> Restart):
  %pip install --user sentence-transformers

In [ ]:
# One-time setup: run this, then Kernel -> Restart, then skip it next time.
%pip install --user sentence-transformers

In [ ]:
# ---------------------------------------------------------------------------
# CELL 1: setup, profile, labels, functions.
# Re-run this cell after editing ARDOOK, LABELS or the thresholds.
# ---------------------------------------------------------------------------

# If the model download fails because the compute node has no internet,
# download the models once from a login node, then uncomment these two lines
# and point HF_HOME at that cache folder. Must come before the imports below.
# import os
# os.environ["HF_HOME"] = "/path/to/your/scratch/hf_cache"

import re
from pathlib import Path

from sentence_transformers import CrossEncoder, SentenceTransformer, util

# Ardook reference profile.
# PLACEHOLDER VALUES - replace these with the real fact sheet.
ARDOOK = {
    "country": "Armenia",
    "legal_type": "for-profit company",
    "stage": "early-stage startup",
    "founded": 2023,
    "women_led": True,
    "focus": "data and AI tools for small businesses",
}

# The three labels from the project doc.
#   queries: phrasings used to FIND the relevant grant sentences (retrieval)
#   claim:   a statement about Ardook the NLI model checks those sentences
#            against (judging)
LABELS = {
    "1) Location": {
        "queries": [
            "Which countries or regions must applicants be located in?",
            "Applicants must be based in or registered in this country.",
        ],
        "claim": (
            f"A {ARDOOK['legal_type']} based in {ARDOOK['country']} "
            f"is eligible to apply."
        ),
    },
    "2) Business type / length of operation": {
        "queries": [
            "What type of organization can apply, nonprofit or for-profit?",
            "How many years must the business have been operating?",
        ],
        "claim": (
            f"A {ARDOOK['stage']} that is a {ARDOOK['legal_type']}, "
            f"founded in {ARDOOK['founded']}, is eligible to apply."
        ),
    },
    "3) Preferred audience": {
        "queries": [
            "Who is this grant intended for?",
            "Priority is given to women-led or underrepresented founders "
            "in a specific sector.",
        ],
        "claim": (
            f"A {'women-led ' if ARDOOK['women_led'] else ''}company working on "
            f"{ARDOOK['focus']} is the kind of applicant this grant is meant for."
        ),
    },
}

EMBED_MODEL = "sentence-transformers/all-MiniLM-L6-v2"
NLI_MODEL = "cross-encoder/nli-MiniLM2-L6-H768"

TOP_K = 3        # evidence sentences kept per label
MIN_SIM = 0.30   # cosine similarity below this counts as "not about this label"


def split_sentences(text, min_words=4):
    """Split grant text into sentences. Handles bullet lists by splitting on
    newlines first, then on sentence-ending punctuation."""
    sentences = []
    for line in text.splitlines():
        line = line.strip(" \t•-*")
        if not line:
            continue
        for s in re.split(r"(?<=[.!?])\s+", line):
            s = s.strip()
            if len(s.split()) >= min_words:
                sentences.append(s)
    return sentences


def find_evidence(embedder, sentence_embs, queries):
    """Return [(sentence_index, similarity)] for the grant sentences closest
    to any of the label's queries."""
    query_embs = embedder.encode(
        queries, convert_to_tensor=True, normalize_embeddings=True
    )
    hits_per_query = util.semantic_search(query_embs, sentence_embs, top_k=TOP_K)

    best = {}  # sentence index -> best similarity across all queries
    for hits in hits_per_query:
        for hit in hits:
            if hit["score"] >= MIN_SIM:
                i = hit["corpus_id"]
                best[i] = max(best.get(i, 0.0), hit["score"])

    return sorted(best.items(), key=lambda pair: -pair[1])[:TOP_K]


def label_order(nli):
    """Read which output column is entailment / contradiction / neutral.
    Falls back to the order documented on the model card."""
    try:
        id2label = {
            int(i): name.lower()
            for i, name in nli.model.config.id2label.items()
        }
        if set(id2label.values()) == {"contradiction", "entailment", "neutral"}:
            return id2label
    except AttributeError:
        pass
    return {0: "contradiction", 1: "entailment", 2: "neutral"}


def judge(nli, id2label, evidence_sentences, claim):
    """Score each evidence sentence (premise) against the Ardook claim
    (hypothesis). Returns one dict of probabilities per sentence."""
    pairs = [(sentence, claim) for sentence in evidence_sentences]
    probs = nli.predict(pairs, apply_softmax=True)
    return [
        {id2label[j]: float(row[j]) for j in range(len(row))}
        for row in probs
    ]


def verdict(scores):
    """Combine sentence-level scores into one verdict for the label.
    The 0-1 'likelihood' is a rough signal, NOT a calibrated probability."""
    top_entail = max(s["entailment"] for s in scores)
    top_contra = max(s["contradiction"] for s in scores)
    likelihood = (top_entail - top_contra + 1) / 2

    if top_contra > 0.5 and top_contra > top_entail:
        return "likely NOT met", likelihood
    if top_entail > 0.5:
        return "likely met", likelihood
    return "unclear - needs a human check", likelihood

In [ ]:
# ---------------------------------------------------------------------------
# CELL 2: load the models. Run once; no need to re-run when you edit Cell 1.
# Both models use the GPU automatically if one is available.
# ---------------------------------------------------------------------------
embedder = SentenceTransformer(EMBED_MODEL)
nli = CrossEncoder(NLI_MODEL)
id2label = label_order(nli)
print("Models loaded. NLI label order:", id2label)

In [ ]:
# ---------------------------------------------------------------------------
# CELL 3: load one grant. Change GRANT_PATH and re-run for each grant.
# Linux paths: use forward slashes.
# ---------------------------------------------------------------------------
GRANT_PATH = "grants/grant_01.txt"

text = Path(GRANT_PATH).read_text(encoding="utf-8", errors="ignore")
sentences = split_sentences(text)
print(f"{len(sentences)} sentences found in {GRANT_PATH}")

sentence_embs = embedder.encode(
    sentences, convert_to_tensor=True, normalize_embeddings=True
)

In [ ]:
# ---------------------------------------------------------------------------
# CELL 4: check each eligibility label. Re-run after changing Cell 1 or 3.
# ---------------------------------------------------------------------------
for label, spec in LABELS.items():
    print("\n" + "=" * 70)
    print(label)
    print("Ardook claim:", spec["claim"])

    evidence = find_evidence(embedder, sentence_embs, spec["queries"])
    if not evidence:
        print("Verdict: not mentioned (may mean no restriction - verify)")
        continue

    evidence_text = [sentences[i] for i, _ in evidence]
    scores = judge(nli, id2label, evidence_text, spec["claim"])
    result, likelihood = verdict(scores)

    print(f"Verdict: {result}   (rough likelihood {likelihood:.2f})")
    for (i, sim), s in zip(evidence, scores):
        print(
            f'  - "{sentences[i]}"\n'
            f"    similarity {sim:.2f} | entail {s['entailment']:.2f} "
            f"| contradict {s['contradiction']:.2f} "
            f"| neutral {s['neutral']:.2f}"
        )